# Uno et al. 2005 Optik: Auto Convention Search: Digitized Aberrations

This notebook computes the digitized aberration quantities defined by Uno et al., *Optik* 116 (2005), formulas (38)-(47), using simulated probe profiles from the Colab GPU smoke-test workflow.

Reference used for the formulas: local PDF `/Users/yuemingguo/Downloads/Uno et al 2005 Optik: Auto Convention Search.pdf`, page 445.

Important notation: Uno's profile width is written with the Greek letter sigma in formula (45). In this notebook it is named `Xigma` so it is not confused with the Gaussian probe smearing parameter `sigma=2` used during probe image generation.


## Auto Convention Search

This copy fits the phase convention automatically for each swept harmonic coefficient. It tries signed harmonic orientations with a fitted constant circular offset, chooses the lowest mean-error mapping, saves the selected convention table, saves the diagnostic plots, and attempts to download the outputs when running in Google Colab.

Latest corrected Colab result summary from `uno_auto_convention_results.zip`: the primary reported phase is `sign * raw_complex_phase / harmonic_order + offset`, wrapped to the coefficient period. The fitted conventions are `A1_value: sign=-1, offset=90 deg, period=180 deg, mean abs error=0.023 deg`; `B2_value/C21: sign=-1, offset=0 deg, period=360 deg, mean abs error<1e-12 deg`; `A2_value: sign=-1, offset=0 deg, period=120 deg, mean abs error<1e-12 deg`; `A3_value: sign=-1, offset=45 deg, period=90 deg, mean abs error<1e-12 deg`; and `S3_value/C32: sign=-1, offset=0 deg, period=180 deg, mean abs error=0.034 deg`. A downloaded JSON may report `360 deg` for `B2_value`, which is equivalent to `0 deg` because its period is `360 deg`.

The fitted offsets are interpreted as a bridge from Uno's digitized coefficients to the simulation convention. The simulator now stores input aberration phases directly as internal angles before evaluating `cos(m * (qphi - angle))`, its CTF uses `exp(+1j * chi)`, and probe formation uses the EM/crystallography inverse transform wrapper `ifft2_em` rather than NumPy/CuPy `ifft2`. The current auto-search sorts by mean error first, then max and median error; this avoids an A2 tie in which the wrong sign has median error `0 deg` but mean error `24 deg` and max error `60 deg`.


## 1. Check GPU


In [ ]:
!nvidia-smi


## 2. Clone or Pull Latest GitHub Code


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/DrYGuo/Aberration-Simulation.git"
ROOT = Path("/content/Aberration-Simulation")

if ROOT.exists():
    print("Repository already exists. Pulling latest main...")
    subprocess.run(["git", "fetch", "origin", "main"], cwd=ROOT, check=True)
    subprocess.run(["git", "reset", "--hard", "origin/main"], cwd=ROOT, check=True)
else:
    print("Cloning repository...")
    subprocess.run(["git", "clone", REPO_URL, str(ROOT)], check=True)

os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, text=True).strip()
print("repo root:", ROOT)
print("commit:", commit)


## 3. Install and Verify Dependencies


In [ ]:
import importlib.util
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

if importlib.util.find_spec("cupy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "cupy-cuda12x"], check=True)
    raise RuntimeError("Installed CuPy. Restart the Colab runtime, then rerun from the top.")

print("Dependencies are ready.")


## 4. Verify CuPy Backend


In [ ]:
!python scripts/check_backend.py

from aberration_simulation.backend import HAS_CUPY, asnumpy, xp

assert HAS_CUPY, "CuPy is not active. Choose a GPU runtime, restart, and rerun."
print("CuPy GPU smoke path is active.")
print("device count:", xp.cuda.runtime.getDeviceCount())


## 5. Run Probe Simulation

This uses the same smoke-test simulation path as `notebooks/colab_gpu_smoke_test.ipynb`.


In [ ]:
!python scripts/run_smoke_test.py


## 6. Uno Formula Implementation

From the local PDF, formulas (38)-(44):

- `Cdf_value = (1/N) sum_k (Xigma_u,k - Xigma_o,k)`
- `A1_value = (2/N) sum_k (Xigma_u,k - Xigma_o,k) exp(2 i theta_k)`
- `B2_value = (2/N) sum_k (Mu_u,k + Mu_o,k) exp(i theta_k)`
- `A2_value = (2/N) sum_k (Mu_u,k + Mu_o,k) exp(3 i theta_k)`
- `Cs_value = (1/N) sum_k (Rho_u,k - Rho_o,k)`
- `S3_value = (2/N) sum_k (Rho_u,k - Rho_o,k) exp(2 i theta_k)`
- `A3_value = (2/N) sum_k (Xigma_u,k - Xigma_o,k) exp(4 i theta_k)`

Formulas (45)-(47) define line-profile characteristics from profile samples `p_j`, where `j=0` is the probe center, `W=sum_j p_j`, and `T=sum_j p_j^2`:

- `Xigma = sqrt((1/W) sum_j j^2 p_j)`
- `Mu = (1/W) sum_j j p_j`
- `Rho = (Xigma^2/T) sum_{j != 0} ((p_j - p_0) p_j / abs(j))`


In [ ]:
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

from aberration_simulation.line_profiles import extract_line_profiles_from_stack
from aberration_simulation.uno_conventions import (
    UNO_HARMONIC_ORDERS,
    add_complex_columns,
    wrap_period_deg,
)

OUTPUT_DIR = ROOT / "outputs"
NPZ_PATH = OUTPUT_DIR / "smoke_probe_images.npz"
CSV_PATH = OUTPUT_DIR / "uno_2005_digitized_aberrations.csv"

UNDER_FOCUS_C1_OFFSET = -909
OVER_FOCUS_C1_OFFSET = 909
PROFILE_RADIUS_PIXELS = 80
PROFILE_STEP_DEGREES = 10
NUM_PROFILE_LINES = int(180 / PROFILE_STEP_DEGREES) + 1

print("under-focus C1_offset:", UNDER_FOCUS_C1_OFFSET, "nm")
print("over-focus C1_offset:", OVER_FOCUS_C1_OFFSET, "nm")
print("line-profile angles: 0 to 170 degrees in 10-degree counter-clockwise steps")


In [ ]:
def load_smoke_outputs(path):
    data = np.load(path, allow_pickle=True)
    names = [str(name) for name in data["parameter_names"]]
    rows = data["parameters"]
    parameters = [
        {name: float(value) for name, value in zip(names, row)}
        for row in rows
    ]
    return data["probe_images"], parameters


COMBINATION_FIELDS = (
    "C1",
    "A1_amp",
    "A1_phase",
    "A2_amp",
    "A2_phase",
    "B2_amp",
    "B2_phase",
    "A3_amp",
    "A3_phase",
    "S3_amp",
    "S3_phase",
    "C3",
)


def combination_key(params):
    return tuple(params.get(field, 0.0) for field in COMBINATION_FIELDS)


def select_under_over_pairs(parameters):
    pairs = {}
    representatives = {}
    for index, params in enumerate(parameters):
        key = combination_key(params)
        representatives.setdefault(key, params)
        pair = pairs.setdefault(key, {})
        if np.isclose(params["C1_offset"], UNDER_FOCUS_C1_OFFSET):
            pair["under"] = index
        if np.isclose(params["C1_offset"], OVER_FOCUS_C1_OFFSET):
            pair["over"] = index

    selected = []
    for key, pair in pairs.items():
        if "under" in pair and "over" in pair:
            selected.append((representatives[key], pair["under"], pair["over"]))
    return selected


probe_images, parameters = load_smoke_outputs(NPZ_PATH)
pairs = select_under_over_pairs(parameters)

print("probe image stack:", probe_images.shape)
print("under/over pairs:", len(pairs))


In [ ]:
def compute_line_characteristics(profiles_np, radius):
    """Compute Xigma, Mu, and Rho for each angular line profile.

    profiles_np has shape `(num_angles, 2 * radius + 1)`.
    The profile-coordinate index j is measured in pixels, with j=0 at center.
    """
    j = np.arange(-radius, radius + 1, dtype=float)
    center_index = int(np.argmin(np.abs(j)))
    p0 = profiles_np[:, center_index]

    W = np.sum(profiles_np, axis=1)
    T = np.sum(profiles_np ** 2, axis=1)
    W = np.where(W == 0, np.nan, W)
    T = np.where(T == 0, np.nan, T)

    Xigma = np.sqrt(np.sum((j[None, :] ** 2) * profiles_np, axis=1) / W)
    Mu = np.sum(j[None, :] * profiles_np, axis=1) / W

    nonzero = j != 0
    curvature_sum = np.sum(
        ((profiles_np[:, nonzero] - p0[:, None]) * profiles_np[:, nonzero])
        / np.abs(j[nonzero])[None, :],
        axis=1,
    )
    Rho = (Xigma ** 2 / T) * curvature_sum

    return {"Xigma": Xigma, "Mu": Mu, "Rho": Rho}


def compute_uno_values(under_chars, over_chars, angles_deg):
    theta = np.deg2rad(angles_deg)
    N = len(theta)

    Xigma_diff = under_chars["Xigma"] - over_chars["Xigma"]
    Mu_sum = under_chars["Mu"] + over_chars["Mu"]
    Rho_diff = under_chars["Rho"] - over_chars["Rho"]

    Cdf_value = np.sum(Xigma_diff) / N
    A1_value = 2 * np.sum(Xigma_diff * np.exp(2j * theta)) / N
    B2_value = 2 * np.sum(Mu_sum * np.exp(1j * theta)) / N
    A2_value = 2 * np.sum(Mu_sum * np.exp(3j * theta)) / N
    Cs_value = np.sum(Rho_diff) / N
    S3_value = 2 * np.sum(Rho_diff * np.exp(2j * theta)) / N
    A3_value = 2 * np.sum(Xigma_diff * np.exp(4j * theta)) / N

    return {
        "Cdf_value": Cdf_value,
        "A1_value": A1_value,
        "B2_value": B2_value,
        "A2_value": A2_value,
        "Cs_value": Cs_value,
        "S3_value": S3_value,
        "A3_value": A3_value,
    }


## 7. Compute Uno Digitized Aberrations


In [ ]:
rows = []
characteristics_by_case = []

for case_index, (params, under_index, over_index) in enumerate(pairs):
    stack = probe_images[:, :, [under_index, over_index]]
    profiles, coords = extract_line_profiles_from_stack(
        stack,
        num_lines=NUM_PROFILE_LINES,
        radius=PROFILE_RADIUS_PIXELS,
    )
    profiles_np = asnumpy(profiles)
    angles_deg = np.asarray(coords["angles_deg"], dtype=float)

    under_chars = compute_line_characteristics(profiles_np[:, :, 0], PROFILE_RADIUS_PIXELS)
    over_chars = compute_line_characteristics(profiles_np[:, :, 1], PROFILE_RADIUS_PIXELS)
    uno_values = compute_uno_values(under_chars, over_chars, angles_deg)

    row = {field: params.get(field, 0.0) for field in COMBINATION_FIELDS}
    row["case_index"] = case_index
    row["under_index"] = under_index
    row["over_index"] = over_index
    row["under_C1_offset"] = UNDER_FOCUS_C1_OFFSET
    row["over_C1_offset"] = OVER_FOCUS_C1_OFFSET
    for name, value in uno_values.items():
        add_complex_columns(row, name, value)
    rows.append(row)

    characteristics_by_case.append({
        "params": params,
        "angles_deg": angles_deg,
        "under_chars": under_chars,
        "over_chars": over_chars,
        "uno_values": uno_values,
    })

print("computed cases:", len(rows))
print("first case Uno values:")
for key, value in characteristics_by_case[0]["uno_values"].items():
    print("  ", key, value)


## 8. Save Results


In [ ]:
CSV_PATH.parent.mkdir(parents=True, exist_ok=True)
fieldnames = list(rows[0].keys())
with CSV_PATH.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print("saved:", CSV_PATH)
print("columns:", len(fieldnames))


## 9. Interpret and Visualize Harmonic Phases

The scalar Uno quantities `Cdf_value` and `Cs_value` do not have angular phase conventions. The harmonic quantities `A1_value`, `B2_value`, `A2_value`, `S3_value`, and `A3_value` do: their raw complex phases are stored as `*_complex_phase_deg` / `*_raw_phase_deg`, while the primary reported `*_phase_deg` columns apply the auto-fitted phase convention for displayed probe images. Line-profile angles increase counter-clockwise in display coordinates. The current fitted convention is `-raw_complex_phase / harmonic_order + offset`, with offsets `A1=90 deg`, `B2=0 deg`, `A2=0 deg`, `A3=45 deg`, and `S3=0 deg`.

These offsets are convention conversions, not changes to Uno formulas `(38)-(44)`. They map the raw complex Uno coefficients onto the simulation convention. Because the simulator phase and Fourier signs have changed, the auto-search should be rerun before copying the fitted offsets into `uno_et_al_2005_optik_latest.ipynb`.


In [ ]:
import json
import zipfile


def circular_difference_deg(a, b, period):
    return np.abs((a - b + period / 2) % period - period / 2)


def circular_mean_deg(values_deg, period_deg):
    values = np.asarray(values_deg, dtype=float)
    angles = 2 * np.pi * values / period_deg
    mean_vector = np.nanmean(np.exp(1j * angles))
    if not np.isfinite(mean_vector):
        return 0.0
    return wrap_period_deg(np.angle(mean_vector, deg=True) * period_deg / 360.0, period_deg)


SCALAR_UNO_COLUMNS = [
    "Cdf_value",
    "Cs_value",
]

PHASE_DIAGNOSTIC_SPECS = [
    {
        "label": "A1",
        "value_name": "A1_value",
        "amp_field": "A1_amp",
        "phase_field": "A1_phase",
    },
    {
        "label": "B2/C21",
        "value_name": "B2_value",
        "amp_field": "B2_amp",
        "phase_field": "B2_phase",
    },
    {
        "label": "A2",
        "value_name": "A2_value",
        "amp_field": "A2_amp",
        "phase_field": "A2_phase",
    },
    {
        "label": "A3",
        "value_name": "A3_value",
        "amp_field": "A3_amp",
        "phase_field": "A3_phase",
    },
    {
        "label": "S3/C32",
        "value_name": "S3_value",
        "amp_field": "S3_amp",
        "phase_field": "S3_phase",
    },
]

AUTO_PHASE_PLOT_PATH = OUTPUT_DIR / "uno_auto_primary_phase_vs_input_all_swept_harmonic_coefficients.png"
AUTO_ERROR_PLOT_PATH = OUTPUT_DIR / "uno_auto_phase_convention_candidate_errors.png"
AUTO_RESULTS_CSV_PATH = OUTPUT_DIR / "uno_2005_digitized_aberrations_auto_conventions.csv"
AUTO_SUMMARY_JSON_PATH = OUTPUT_DIR / "uno_auto_phase_convention_summary.json"
AUTO_OUTPUT_ZIP_PATH = OUTPUT_DIR / "uno_auto_convention_results.zip"


def candidate_phase(raw_complex_phase_deg, order, sign, offset_deg):
    period = 360.0 / order
    return np.mod(sign * raw_complex_phase_deg / order + offset_deg, period)


def fit_signed_offset_convention(rows, spec):
    value_name = spec["value_name"]
    amp_field = spec["amp_field"]
    phase_field = spec["phase_field"]
    order = UNO_HARMONIC_ORDERS[value_name]
    period = 360.0 / order

    selected_rows = [row for row in rows if not np.isclose(row.get(amp_field, 0.0), 0.0)]
    if not selected_rows:
        return None, []

    raw = np.asarray([row[value_name + "_complex_phase_deg"] for row in selected_rows], dtype=float)
    target = np.asarray([row[phase_field] % period for row in selected_rows], dtype=float)

    fits = []
    for sign in (1.0, -1.0):
        base = np.mod(sign * raw / order, period)
        offset = circular_mean_deg(target - base, period)
        predicted = np.mod(base + offset, period)
        errors = circular_difference_deg(predicted, target, period)
        fits.append({
            "label": spec["label"],
            "value_name": value_name,
            "amp_field": amp_field,
            "phase_field": phase_field,
            "order": order,
            "period_deg": period,
            "sign": sign,
            "offset_deg": float(offset),
            "mean_abs_error_deg": float(np.nanmean(errors)),
            "median_abs_error_deg": float(np.nanmedian(errors)),
            "max_abs_error_deg": float(np.nanmax(errors)),
        })

    best = sorted(fits, key=lambda item: (item["mean_abs_error_deg"], item["max_abs_error_deg"], item["median_abs_error_deg"]))[0]

    diagnostics = []
    for row in selected_rows:
        predicted = candidate_phase(row[value_name + "_complex_phase_deg"], order, best["sign"], best["offset_deg"])
        target_value = row[phase_field] % period
        diagnostics.append({
            "label": spec["label"],
            "value_name": value_name,
            "case_index": row["case_index"],
            "amp": row[amp_field],
            "input_phase_deg": row[phase_field],
            "input_phase_mod_period_deg": target_value,
            "period_deg": period,
            "complex_phase_deg_raw": row[value_name + "_complex_phase_deg"],
            "auto_phase_deg": float(predicted),
            "auto_phase_error_deg": float(circular_difference_deg(predicted, target_value, period)),
            "auto_sign": best["sign"],
            "auto_offset_deg": best["offset_deg"],
        })
    return best, diagnostics


auto_convention_summary = []
auto_diagnostics_by_coeff = {}
for spec in PHASE_DIAGNOSTIC_SPECS:
    best, diagnostics = fit_signed_offset_convention(rows, spec)
    auto_diagnostics_by_coeff[spec["label"]] = diagnostics
    if best is not None:
        auto_convention_summary.append(best)

print("Auto-fitted harmonic phase conventions:")
for item in auto_convention_summary:
    print(item)

# Write auto-fitted phase columns back onto row dictionaries.
for row in rows:
    for item in auto_convention_summary:
        value_name = item["value_name"]
        predicted = candidate_phase(
            row[value_name + "_complex_phase_deg"],
            item["order"],
            item["sign"],
            item["offset_deg"],
        )
        row[value_name + "_auto_phase_deg"] = float(predicted)
        row[value_name + "_auto_phase_sign"] = float(item["sign"])
        row[value_name + "_auto_phase_offset_deg"] = float(item["offset_deg"])
        row[value_name + "_phase_deg"] = float(predicted)
        row[value_name + "_phase_convention"] = "auto_signed_offset"

# Save auto-fitted CSV with all rows.
fieldnames = list(rows[0].keys())
with AUTO_RESULTS_CSV_PATH.open("w", newline="") as handle:
    writer = csv.DictWriter(handle, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

with AUTO_SUMMARY_JSON_PATH.open("w") as handle:
    json.dump(auto_convention_summary, handle, indent=2)

active_specs = [
    spec for spec in PHASE_DIAGNOSTIC_SPECS
    if auto_diagnostics_by_coeff[spec["label"]]
]

if active_specs:
    print("Rendering: Primary reported phase vs input phase for all swept harmonic coefficients")
    fig, axes = plt.subplots(1, len(active_specs), figsize=(4.8 * len(active_specs), 4.2), squeeze=False)
    for axis, spec in zip(axes.ravel(), active_specs):
        diagnostics = auto_diagnostics_by_coeff[spec["label"]]
        period = diagnostics[0]["period_deg"]
        target = np.asarray([item["input_phase_mod_period_deg"] for item in diagnostics])
        primary = np.asarray([item["auto_phase_deg"] for item in diagnostics])
        amp = np.asarray([item["amp"] for item in diagnostics])
        sc = axis.scatter(target, primary, c=amp, cmap="viridis", s=28, alpha=0.9)
        axis.plot([0, period], [0, period], color="black", linewidth=1, linestyle="--")
        pad = 0.04 * period
        axis.set_title(spec["label"])
        axis.set_xlabel("input phase mod period (deg)")
        axis.set_ylabel(spec["value_name"] + "_auto_phase_deg")
        axis.set_xlim(-pad, period + pad)
        axis.set_ylim(-pad, period + pad)
        axis.grid(alpha=0.3)
        fig.colorbar(sc, ax=axis, label=spec["amp_field"])
    fig.suptitle("Primary reported phase vs input phase for all swept harmonic coefficients")
    fig.tight_layout()
    fig.savefig(AUTO_PHASE_PLOT_PATH, dpi=180, bbox_inches="tight")
    print("saved auto primary phase plot:", AUTO_PHASE_PLOT_PATH)
    plt.show()

if active_specs:
    print("Rendering: auto convention fit errors")
    labels = [item["label"] for item in auto_convention_summary]
    medians = [item["median_abs_error_deg"] for item in auto_convention_summary]
    means = [item["mean_abs_error_deg"] for item in auto_convention_summary]
    maxes = [item["max_abs_error_deg"] for item in auto_convention_summary]
    x = np.arange(len(labels))
    fig, ax = plt.subplots(figsize=(10, 4.5))
    ax.bar(x - 0.25, medians, width=0.25, label="median")
    ax.bar(x, means, width=0.25, label="mean")
    ax.bar(x + 0.25, maxes, width=0.25, label="max")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=20, ha="right")
    ax.set_ylabel("absolute phase error (deg)")
    ax.set_title("Auto-fitted convention errors")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(AUTO_ERROR_PLOT_PATH, dpi=180, bbox_inches="tight")
    print("saved auto error plot:", AUTO_ERROR_PLOT_PATH)
    plt.show()

with zipfile.ZipFile(AUTO_OUTPUT_ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in [AUTO_RESULTS_CSV_PATH, AUTO_SUMMARY_JSON_PATH, AUTO_PHASE_PLOT_PATH, AUTO_ERROR_PLOT_PATH]:
        if path.exists():
            zf.write(path, arcname=path.name)
print("saved auto convention output zip:", AUTO_OUTPUT_ZIP_PATH)

try:
    from google.colab import files
    files.download(str(AUTO_OUTPUT_ZIP_PATH))
except Exception as exc:
    print("Colab download skipped:", exc)


## 9. Quick Visual Check


In [ ]:
def find_first_case(**criteria):
    for index, item in enumerate(characteristics_by_case):
        params = item["params"]
        if all(np.isclose(params.get(key, 0.0), value) for key, value in criteria.items()):
            return index, item
    return 0, characteristics_by_case[0]


case_index, item = find_first_case(A1_amp=60, A1_phase=157.5)
angles_deg = item["angles_deg"]
row = rows[case_index]

fig, axes = plt.subplots(3, 1, figsize=(8, 9), sharex=True)
for axis, metric in zip(axes, ["Xigma", "Mu", "Rho"]):
    axis.plot(angles_deg, item["under_chars"][metric], marker="o", label="under")
    axis.plot(angles_deg, item["over_chars"][metric], marker="s", label="over")
    axis.set_ylabel(metric)
    axis.grid(True, alpha=0.3)
    axis.legend()
axes[-1].set_xlabel("line angle (deg)")
fig.suptitle("Uno line characteristics, case {}".format(case_index))
fig.tight_layout()
plt.show()

print("case parameters:", item["params"])
print("Uno values:")
for key, value in item["uno_values"].items():
    primary_phase = row.get(key + "_phase_deg")
    raw_phase = row.get(key + "_complex_phase_deg")
    period = row.get(key + "_phase_period_deg")
    print("  {} = {} | abs={} primary_phase_deg={} raw_complex_phase_deg={} period={}".format(
        key,
        value,
        np.abs(value),
        primary_phase,
        raw_phase,
        period,
    ))
